# Starter TF-IDF + Logistic Regression Baseline

In [1]:
# Starter Baseline — TF-IDF + Logistic Regression
# No transformers, no LLM embeddings.

from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression

DATA_DIR = Path("/kaggle/input/YOUR_MATH_TOPIC_DATASET")  # CHANGE THIS
SEED = 2026
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print(train.shape, test.shape)
display(train.head())

def normalize_math_text(s):
    s = str(s)
    s = s.replace("\\left", " ").replace("\\right", " ")
    s = s.replace("\\,", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.lower().strip()

train["text"] = train["Question"].map(normalize_math_text)
test["text"] = test["Question"].map(normalize_math_text)
X = train["text"]
y = train["label"].astype(int).values
X_test = test["text"]

features = FeatureUnion([
    ("word", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b|\\[a-zA-Z]+|[=+\-*/^<>≤≥]+",
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 6),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
])
model = LogisticRegression(C=4.0, max_iter=3000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=SEED)
pipe = Pipeline([("features", features), ("model", model)])

tr_idx, val_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=SEED, stratify=y)
pipe.fit(X.iloc[tr_idx], y[tr_idx])
val_pred = pipe.predict(X.iloc[val_idx])
print("Validation macro F1:", f1_score(y[val_idx], val_pred, average="macro"))
print(classification_report(y[val_idx], val_pred, digits=4))
print(pd.DataFrame(confusion_matrix(y[val_idx], val_pred)))

pipe.fit(X, y)
pred = pipe.predict(X_test)
submission = pd.DataFrame({"id": test["id"], "label": pred.astype(int)})
submission.to_csv("/kaggle/working/submission.csv", index=False)
display(submission.head())
print("Saved /kaggle/working/submission.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/YOUR_MATH_TOPIC_DATASET/train.csv'